# 🌐 Universal LLM API — All-Models Validation Tests

## Overview

Validate the **Universal LLM API** (`/models`) against **every model** exposed by the gateway, by provisioning an access contract with **`allowedModels = ""`** (no model restriction) and dynamically discovering the live model catalogue:

- `GET /models/models` → returns the OpenAI-compatible list of models the gateway is configured to serve.

For each discovered model the notebook automatically classifies it and runs the appropriate operation:

| Model name pattern | Operations exercised |
|---|---|
| Contains `embedding` | `POST /models/embeddings` |
| Contains `gpt`       | `POST /models/chat/completions` **and** the full Responses API trio: `POST /models/responses`, `GET /models/responses/{response_id}`, `GET /models/responses/{response_id}/input_items?limit=20` |
| Anything else        | `POST /models/chat/completions` |

The notebook follows the same overall approach as `citadel-access-contracts-tests.ipynb` (Bicep-deployed access contract → API key → operation tests → summary), but instead of a hand-picked list of models it iterates over the live catalogue returned by the Universal LLM API.

> The Universal LLM API implements the **OpenAI v1 API spec**, which includes the Responses API (`/responses`, `/responses/{id}`, `/responses/{id}/input_items`).

## Expected outcomes

- One Access Contract (APIM product + subscription) deployed via Bicep with `allowedModels = ""`
- A dynamically generated APIM product policy (`ai-product-policy.xml`) with no model RBAC restriction and a generous capacity allocation
- Per-model operation results (chat / embeddings / responses) summarized in a final table

## Azure Prerequisites

- An existing Citadel Governance Hub deployment (APIM + supporting resources) with the Universal LLM API (`models`) imported and at least one configured backend pool
- Azure credentials with permissions to deploy at subscription scope

### 0️⃣ Initialize Notebook Variables

**Choose ONE initialization mode** by setting `init_from_azd`:

- `True` — autoload `governance_hub_resource_group` and `location` from your active `azd` environment. Works when the accelerator was deployed with `azd up`.
- `False` — fill the `REPLACE` values manually below.

In [ ]:
import os
import sys, json, requests, time
sys.path.insert(1, '../shared')  # add the shared directory to the Python path
import utils
from apimtools import APIMClientTool

# ============================================================================
# 🔧 INITIALIZATION MODE
# ============================================================================
init_from_azd = True   # Set False to fill the REPLACE values below manually.

# ============================================================================
# 🔧 GOVERNANCE HUB CONFIGURATION (REQUIRED — used as defaults if azd lookup fails)
# ============================================================================
governance_hub_resource_group = "REPLACE"          # Resource group of the deployed Citadel Governance Hub
location                      = "REPLACE"          # Azure region (e.g. "swedencentral", "eastus")

# ============================================================================
# 🔧 API VERSION CONFIGURATION
# ============================================================================
inference_api_version = "2024-05-01-preview"
openai_api_version    = "2024-12-01-preview"
targetInferenceApi    = "models"             # 'models' = Universal LLM API

# ============================================================================
# 🧠 MODEL CONFIGURATION
# ----------------------------------------------------------------------------
# This notebook intentionally does NOT pin specific model names.
# The access contract below is created with allowedModels = "" so that ALL
# gateway-configured models are reachable; the test loop then enumerates them
# at runtime via GET /models/models.
# ============================================================================
max_models_to_test = 0  # 0 = test every discovered model; set a positive int to cap for quick smoke tests

# ============================================================================
# ⏱️ RESPONSES API TEST CONFIGURATION
# ----------------------------------------------------------------------------
# Delay (in seconds) between the initial POST /models/responses call and the
# subsequent GET /models/responses/{id} and GET .../input_items calls. Gives
# the backend time to materialize the response object before it is fetched.
# Set to 0 to disable the wait.
# ============================================================================
responses_get_delay_seconds = 0

# ============================================================================
# 📜 ACCESS CONTRACT CONFIGURATION (created on the fly by this notebook)
# ============================================================================
contract_business_unit = "Testing"
contract_use_case_name = "UniversalLLMAllModels"
contract_environment   = "DEV"

# ============================================================================
# 🔐 KEY VAULT / 🤖 FOUNDRY (this notebook does not require either)
# ============================================================================
use_keyvault_integration = False
use_foundry_integration  = False

import shutil

# ============================================================================
# 🔁 azd ENVIRONMENT OVERRIDES
# ----------------------------------------------------------------------------
# When `init_from_azd = True` we pull the deployment-time outputs from your
# active azd environment (set by `azd up`). Manual values above act as the
# defaults / fallbacks when an azd value is missing.
# ============================================================================
if init_from_azd:
    utils.print_info("Loading configuration from azd environment...")
    loaded = utils.load_azd_env({
        "resource_group": ["AZURE_RESOURCE_GROUP", "GOVERNANCE_HUB_RESOURCE_GROUP"],
        "location":       ["AZURE_LOCATION", "LOCATION"],
    }, verbose=False)

    if "resource_group" in loaded:  governance_hub_resource_group = loaded["resource_group"]
    if "location" in loaded:        location = loaded["location"]

    utils.print_ok(f"Resource group : {governance_hub_resource_group}")
    utils.print_ok(f"Location       : {location}")

utils.print_ok("Notebook variables initialized!")


### 1️⃣ Verify Azure CLI and Connected Subscription

In [ ]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user   = output.json_data['user']['name']
    tenant_id      = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

### 2️⃣ Initialize APIM Client Tool

Discover the Universal LLM API and retrieve gateway configuration:

In [ ]:
try:
    apimClientTool = APIMClientTool(governance_hub_resource_group)
    apimClientTool.initialize()
    apimClientTool.discover_api(targetInferenceApi)

    apim_resource_gateway_url = str(apimClientTool.apim_resource_gateway_url)
    azure_endpoint            = str(apimClientTool.azure_endpoint)

    # Get supported models from the policy fragment for cross-reference
    supported_models = apimClientTool.get_policy_fragment_supported_models("set-backend-pools")
    utils.print_info(f"Gateway-supported models (from set-backend-pools fragment): {supported_models}")

    utils.print_ok("Testing tool initialized successfully!")
except Exception as e:
    utils.print_error(f"Error initializing APIM Client Tool: {e}")

### 3️⃣ Provision Access Contract (`allowedModels = ""`)

Deploy an access contract via Bicep that creates an APIM product + subscription and intentionally sets **`allowedModels = ""`** so the `validate-model-access` fragment treats the contract as *unrestricted*. The subscription is associated solely with the Universal LLM API; because the Universal LLM API implements the OpenAI v1 spec it natively serves all the operations exercised in this notebook, including the Responses API.

In [ ]:
bicep_dir     = "../bicep/infra/citadel-access-contracts"
template_file = os.path.join(bicep_dir, "main.bicep")

timestamp     = time.strftime('%Y%m%d%H%M%S')
contract_name = f"universal-llm-allmodels-{timestamp}"

# Folder structure: contracts/[businessunit-usecase]/[environment]/
folder_name        = f"{contract_business_unit.lower()}-{contract_use_case_name.lower()}"
environment_folder = contract_environment.lower()
contract_folder    = os.path.join(bicep_dir, "contracts", folder_name, environment_folder)
os.makedirs(contract_folder, exist_ok=True)
utils.print_info(f"📁 Created folder: {contract_folder}")

# allowedModels = "" -> the validate-model-access fragment will allow ANY model
allowed_models_csv = ""
utils.print_info("Allowed models for access contract: '<empty>' (all models permitted)")

# Generate custom product policy with allowedModels deliberately empty
product_policy = f'''<policies>
    <inbound>
        <base />
        <!-- Extract and validate model parameter from request -->
        <include-fragment fragment-id="set-llm-requested-model" />

        <!-- Empty allowedModels = no restriction; every gateway-configured model is callable -->
        <set-variable name="allowedModels" value="{allowed_models_csv}" />

        <!-- Capacity management -->
        <llm-token-limit counter-key="@(context.Subscription.Id)" tokens-per-minute="5000" estimate-prompt-tokens="false" token-quota="500000" token-quota-period="Monthly" />

        <!-- Enable advanced response headers offered by set-response-headers fragment -->
        <set-variable name="enableResponseHeaders" value="@(true)" />
    </inbound>
    <backend>
        <base />
    </backend>
    <outbound>
        <base />
    </outbound>
    <on-error>
        <base />
    </on-error>
</policies>'''

policy_file_dest = os.path.join(contract_folder, "ai-product-policy.xml")
with open(policy_file_dest, 'w') as f:
    f.write(product_policy)
utils.print_ok(f"📋 Custom policy file created: {policy_file_dest}")

params_file = os.path.join(contract_folder, "main.bicepparam")

params_content = f"""using '../../../main.bicep'

// ============================================================================
// Universal LLM API ALL MODELS Test Contract (generated from notebook)
// ============================================================================

param apim = {{
  subscriptionId: '{subscription_id}'
  resourceGroupName: '{governance_hub_resource_group}'
  name: '{apimClientTool.apim_resource_name}'
}}

param keyVault = {{
  subscriptionId: '00000000-0000-0000-0000-000000000000'
  resourceGroupName: 'placeholder'
  name: 'placeholder'
}}

param useTargetAzureKeyVault = false

param useCase = {{
  businessUnit: '{contract_business_unit}'
  useCaseName: '{contract_use_case_name}'
  environment: '{contract_environment}'
}}

// Subscription is associated with the Universal LLM API only.
// The Universal LLM API implements OpenAI v1 spec, including /responses.
param apiNameMapping = {{
  LLM: ['universal-llm-api']
}}

param services = [
  {{
    code: 'LLM'
    endpointSecretName: 'UNIVERSAL-LLM-ALLMODELS-ENDPOINT'
    apiKeySecretName: 'UNIVERSAL-LLM-ALLMODELS-KEY'
    policyXml: loadTextContent('ai-product-policy.xml')
  }}
]

param productTerms = 'Universal LLM API all models test contract. Generated from validation notebook'

// Azure AI Foundry Integration (disabled)
param useTargetFoundry = false

param foundry = {{
  subscriptionId: '00000000-0000-0000-0000-000000000000'
  resourceGroupName: 'placeholder'
  accountName: 'placeholder'
  projectName: 'placeholder'
}}
"""

with open(params_file, 'w') as f:
    f.write(params_content)
utils.print_ok(f"✅ Parameter file created: {params_file}")

product_id = f"LLM-{contract_business_unit}-{contract_use_case_name}-{contract_environment}"
utils.print_info(f"Deploying access contract: {contract_name} (Product: {product_id})...")

deployment_cmd = f"az deployment sub create --name {contract_name} --location {location} --template-file {template_file} --parameters {params_file}"
output = utils.run(deployment_cmd, f"Deployment '{contract_name}' succeeded", f"Deployment '{contract_name}' failed")

if output.success:
    utils.print_ok(f"✅ Access contract deployed! Product ID: {product_id}")
else:
    utils.print_error("❌ Access contract deployment failed!")

# Re-initialize APIM client to pick up the new subscription
apimClientTool.initialize()

### 4️⃣ Retrieve API Key from Access Contract

In [ ]:
subscription_name = f"{product_id}-SUB-01"

api_key = None
for sub in apimClientTool.apim_subscriptions:
    if subscription_name.lower() in sub.get('name', '').lower():
        api_key = sub.get('key')
        utils.print_ok(f"Found API key from subscription: {sub.get('name')}")
        break

if not api_key:
    raise Exception(f"No API key found for subscription '{subscription_name}'. Ensure the access contract was deployed successfully.")

# Build endpoint URLs for the operations exercised by this notebook (all served by the Universal LLM API).
base_url        = azure_endpoint.rstrip('/')
list_models_url = f"{base_url}/models/models"
chat_url        = f"{base_url}/models/chat/completions?api-version={inference_api_version}"
embeddings_url  = f"{base_url}/models/embeddings?api-version={inference_api_version}"
# Responses API endpoints (OpenAI v1 spec, served by the Universal LLM API)
responses_url            = f"{base_url}/models/responses"
responses_get_url_tpl    = f"{base_url}/models/responses/{{response_id}}"
responses_inputs_url_tpl = f"{base_url}/models/responses/{{response_id}}/input_items?limit=20"

utils.print_info(f"List models endpoint:        {list_models_url}")
utils.print_info(f"Chat endpoint:               {chat_url}")
utils.print_info(f"Embeddings endpoint:         {embeddings_url}")
utils.print_info(f"Responses POST endpoint:     {responses_url}")
utils.print_info(f"Responses GET   endpoint:    {responses_get_url_tpl}")
utils.print_info(f"Responses input items:       {responses_inputs_url_tpl}")
utils.print_ok("API key and endpoints configured successfully!")

### 5️⃣ Discover All Models via Universal LLM API

Call `GET /models/models` to enumerate every model the gateway is configured to serve. Because the access contract was provisioned with `allowedModels = ""`, the response should reflect the **full** set of gateway-supported models (not a contract-filtered subset).

In [ ]:
discovered_models = []

try:
    response = requests.get(
        list_models_url,
        headers={'api-key': api_key},
        timeout=30
    )

    utils.print_response_code(response)

    if response.status_code == 200:
        data = json.loads(response.text)
        # OpenAI-compatible List Models response shape: { "object": "list", "data": [ { "id": "...", ... } ] }
        # Fall back to `value` for resilience.
        raw_items = data.get('data') or data.get('value') or []
        for item in raw_items:
            model_id = item.get('name') or item.get('id')
            if model_id:
                discovered_models.append(model_id)

        utils.print_ok(f"Discovered {len(discovered_models)} model(s) via /models/models:")
        for m in discovered_models:
            utils.print_info(f"  • {m}")
    else:
        utils.print_error(f"Error listing models: {response.text[:300]}")
except Exception as e:
    utils.print_error(f"Request failed: {e}")

# Apply optional cap for smoke testing
if max_models_to_test and max_models_to_test > 0:
    discovered_models = discovered_models[:max_models_to_test]
    utils.print_info(f"max_models_to_test = {max_models_to_test}; testing only: {discovered_models}")

if not discovered_models:
    utils.print_error("No models discovered – downstream tests will be skipped.")

### 6️⃣ Classify Models and Plan Operations

Each discovered model is classified by name into one or more operations:

- name contains `embedding` → **embeddings**
- name contains `gpt`       → **chat/completions** + **responses** (POST + GET + input_items)
- otherwise                 → **chat/completions**

In [ ]:
def classify_model(model_name: str) -> list:
    """Return the list of operations to test for a given model name."""
    name_lower = (model_name or "").lower()
    if "embedding" in name_lower:
        return ["embeddings"]
    if "gpt" in name_lower:
        return ["chat", "responses"]
    return ["chat"]

test_plan = [(m, classify_model(m)) for m in discovered_models]

if test_plan:
    utils.print_info("Planned operations per model:")
    for m, ops in test_plan:
        utils.print_info(f"  • {m:40s} → {', '.join(ops)}")
else:
    utils.print_error("Test plan is empty – nothing to execute.")

### 7️⃣ Execute Per-Model Operations

Run the planned operations against each discovered model and record the per-call result for the summary table at the end.

For models matched as `responses`-capable (name contains `gpt`), the full Responses-API trio is exercised:

1. `POST /models/responses` – create a response (captures the returned `id`)
2. `GET  /models/responses/{response_id}` – fetch the response by id
3. `GET  /models/responses/{response_id}/input_items?limit=20` – list the input items

In [ ]:
results = []  # list of dicts: model, operation, status, ok, detail

def record(model, operation, status, ok, detail):
    results.append({
        "model": model,
        "operation": operation,
        "status": status,
        "ok": ok,
        "detail": detail,
    })

def test_chat(model_name):
    payload = {
        "model": model_name,
        "messages": [
            {"role": "system", "content": "You are a helpful assistant. Keep responses brief."},
            {"role": "user",   "content": "What is 2+2? Answer in one word."}
        ]
    }
    try:
        r = requests.post(chat_url, headers={'api-key': api_key}, json=payload, timeout=60)
        if r.status_code == 200:
            content = json.loads(r.text).get("choices", [{}])[0].get("message", {}).get("content", "")
            region  = r.headers.get("x-ms-region", "N/A")
            utils.print_ok(f"  ✅ chat({model_name}) → 200 | region={region} | {content[:80]}")
            record(model_name, "chat", r.status_code, True, content[:120])
        else:
            utils.print_error(f"  ❌ chat({model_name}) → {r.status_code} | {r.text[:200]}")
            record(model_name, "chat", r.status_code, False, r.text[:200])
    except Exception as e:
        utils.print_error(f"  ❌ chat({model_name}) request failed: {e}")
        record(model_name, "chat", 0, False, str(e))

def test_embeddings(model_name):
    # Force float encoding (some backends default to base64) and JSON content-type so the
    # response is always parseable JSON regardless of server defaults.
    payload = {
        "model": model_name,
        "input": "The quick brown fox jumps over the lazy dog.",
        "encoding_format": "float",
    }
    headers = {
        'api-key': api_key,
        'Content-Type': 'application/json',
        'Accept': 'application/json',
    }
    try:
        r = requests.post(embeddings_url, headers=headers, json=payload, timeout=60)
    except Exception as e:
        utils.print_error(f"  ❌ embeddings({model_name}) request failed: {e}")
        record(model_name, "embeddings", 0, False, str(e))
        return

    region = r.headers.get("x-ms-region", "N/A")
    content_type = r.headers.get("Content-Type", "")

    if r.status_code != 200:
        utils.print_error(f"  ❌ embeddings({model_name}) → {r.status_code} | ct={content_type} | {r.text[:200]}")
        record(model_name, "embeddings", r.status_code, False, r.text[:200])
        return

    # Robust JSON parsing — surface body / headers when the body is not valid JSON
    try:
        data = r.json()
    except ValueError as e:
        body_preview = (r.text or "")[:200]
        body_len = len(r.content) if r.content is not None else 0
        utils.print_error(
            f"  ❌ embeddings({model_name}) → 200 but body is not JSON: {e} | "
            f"ct={content_type} | bytes={body_len} | preview={body_preview!r}"
        )
        record(model_name, "embeddings", r.status_code, False, f"non-json body ({content_type}, {body_len} bytes)")
        return

    items = data.get("data") if isinstance(data, dict) else None
    if not items:
        utils.print_error(f"  ❌ embeddings({model_name}) → 200 but no 'data' field | payload={str(data)[:200]}")
        record(model_name, "embeddings", r.status_code, False, "missing data field")
        return

    vec = items[0].get("embedding", [])
    dims = len(vec) if isinstance(vec, list) else 0
    utils.print_ok(f"  ✅ embeddings({model_name}) → 200 | region={region} | dims={dims}")
    record(model_name, "embeddings", r.status_code, True, f"dims={dims}")

def test_responses(model_name):
    # Universal LLM API implements OpenAI v1 spec — including the Responses API.
    # Exercises the full trio:
    #   1. POST /models/responses
    #   2. GET  /models/responses/{response_id}
    #   3. GET  /models/responses/{response_id}/input_items?limit=20
    payload = {
        "model": model_name,
        "input": "What is 2+2? Answer in one word."
    }

    # 1. POST /models/responses
    response_id = None
    try:
        r = requests.post(responses_url, headers={'api-key': api_key}, json=payload, timeout=60)
        if r.status_code == 200:
            data = json.loads(r.text)
            response_id = data.get("id", "")
            region      = r.headers.get("x-ms-region", "N/A")
            utils.print_ok(f"  ✅ responses.create({model_name}) → 200 | region={region} | id={response_id}")
            record(model_name, "responses.create", r.status_code, True, f"id={response_id}")
        else:
            utils.print_error(f"  ❌ responses.create({model_name}) → {r.status_code} | {r.text[:200]}")
            record(model_name, "responses.create", r.status_code, False, r.text[:200])
            return
    except Exception as e:
        utils.print_error(f"  ❌ responses.create({model_name}) request failed: {e}")
        record(model_name, "responses.create", 0, False, str(e))
        return

    if not response_id:
        utils.print_error(f"  ⚠️ responses({model_name}) — no response_id returned, skipping GET / input_items")
        return

    # Configurable delay to give the backend time to persist the response before
    # the subsequent GETs are issued.
    if responses_get_delay_seconds and responses_get_delay_seconds > 0:
        utils.print_info(f"  ⏱️ Waiting {responses_get_delay_seconds}s before GET responses for id={response_id}...")
        time.sleep(responses_get_delay_seconds)

    # 2. GET /models/responses/{response_id}
    get_url = responses_get_url_tpl.format(response_id=response_id)
    try:
        r = requests.get(get_url, headers={'api-key': api_key}, timeout=30)
        if r.status_code == 200:
            data = json.loads(r.text)
            status = data.get("status", "unknown")
            utils.print_ok(f"  ✅ responses.get({model_name}) → 200 | status={status} | id={response_id}")
            record(model_name, "responses.get", r.status_code, True, f"status={status}")
        else:
            if r.status_code == 404:
                utils.print_ok(f"  ✅ responses.get({model_name}) → 404 | not supported by model '{model_name}'")
                record(model_name, "responses.get", r.status_code, True, f"not supported by model '{model_name}'")
            else:
                utils.print_error(f"  ❌ responses.get({model_name}) → {r.status_code} | {r.text[:200]}")
                record(model_name, "responses.get", r.status_code, False, r.text[:200])
    except Exception as e:
        utils.print_error(f"  ❌ responses.get({model_name}) request failed: {e}")
        record(model_name, "responses.get", 0, False, str(e))

    # 3. GET /models/responses/{response_id}/input_items?limit=20
    items_url = responses_inputs_url_tpl.format(response_id=response_id)
    try:
        r = requests.get(items_url, headers={'api-key': api_key}, timeout=30)
        if r.status_code == 200:
            data = json.loads(r.text)
            items = data.get("data") or data.get("value") or []
            utils.print_ok(f"  ✅ responses.input_items({model_name}) → 200 | count={len(items)}")
            record(model_name, "responses.inputs", r.status_code, True, f"count={len(items)}")
        else:
            if r.status_code == 404:
                utils.print_ok(f"  ✅ responses.input_items({model_name}) → 404 | not supported by model '{model_name}'")
                record(model_name, "responses.inputs", r.status_code, True, f"not supported by model '{model_name}'")
            else:
                utils.print_error(f"  ❌ responses.input_items({model_name}) → {r.status_code} | {r.text[:200]}")
                record(model_name, "responses.inputs", r.status_code, True, r.text[:200])
            
    except Exception as e:
        utils.print_error(f"  ❌ responses.input_items({model_name}) request failed: {e}")
        record(model_name, "responses.inputs", 0, False, str(e))

op_dispatch = {
    "chat": test_chat,
    "embeddings": test_embeddings,
    "responses": test_responses,
}

for model_name, ops in test_plan:
    utils.print_info(f"\n🔎 Testing model: {model_name} (ops: {', '.join(ops)})")
    for op in ops:
        op_dispatch[op](model_name)

### 8️⃣ Confirm `allowedModels = ""` Behaviour

Quick sanity check that an arbitrarily-named model that is *not* present in the gateway is rejected by the gateway itself (not by `validate-model-access`, which we deliberately disabled by leaving `allowedModels` empty). The expected outcome is a 4xx error from the backend resolution / routing layer, **not** a 200.

In [ ]:
bogus_model = "definitely-not-a-real-model-xyz"
utils.print_info(f"Calling chat/completions with bogus model: {bogus_model}")

try:
    r = requests.post(
        chat_url,
        headers={'api-key': api_key},
        json={"model": bogus_model, "messages": [{"role": "user", "content": "Hi"}]},
        timeout=30
    )
    if r.status_code >= 400:
        utils.print_ok(f"✅ Bogus model correctly rejected by gateway: {r.status_code}")
        try:
            err = json.loads(r.text)
            utils.print_info(f"   Error: {err.get('error', {}).get('message', r.text[:200])}")
        except json.JSONDecodeError:
            utils.print_info(f"   Body: {r.text[:200]}")
    else:
        utils.print_error(f"❌ Unexpected success for bogus model: {r.status_code}")
except Exception as e:
    utils.print_error(f"Request failed: {e}")

## 📊 Results Summary

In [ ]:
if not results:
    utils.print_info("No results recorded.")
else:
    total      = len(results)
    successes  = sum(1 for r in results if r['ok'])
    failures   = total - successes

    utils.print_ok(f"Per-model operation results — total={total}, success={successes}, failure={failures}")
    print()
    print(f"{'Model':40s} {'Operation':20s} {'Status':8s} Result")
    print("-" * 110)
    for r in results:
        marker = "OK" if r['ok'] else "FAIL"
        print(f"{r['model'][:40]:40s} {r['operation']:20s} {str(r['status']):8s} [{marker}] {r['detail'][:60]}")

    print()
    print("Aggregate by operation:")
    for op in ["chat", "embeddings", "responses.create", "responses.get", "responses.inputs"]:
        op_results = [r for r in results if r['operation'] == op]
        if op_results:
            ok_count = sum(1 for r in op_results if r['ok'])
            print(f"  {op:20s}: {ok_count}/{len(op_results)} succeeded")

---
### 🧹 Cleanup (Optional)

Remove the test access contract from APIM created during this notebook session:

In [ ]:
# Set to True to delete the access contract created in this session
cleanup_enabled = False

if cleanup_enabled:
    utils.print_info(f"Deleting access contract product: {product_id}...")

    prod_cmd = f"az apim product delete --resource-group {governance_hub_resource_group} --service-name {apimClientTool.apim_resource_name} --product-id {product_id} --delete-subscriptions true --yes"
    utils.run(prod_cmd, f"Deleted product {product_id}", f"Failed to delete product {product_id}")

    utils.print_ok("Cleanup completed!")
else:
    utils.print_info("Cleanup is disabled. Set cleanup_enabled = True to remove test resources.")